In [ ]:
import sqlite3
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from itertools import combinations
import re

spark = SparkSession.builder.appName("FrequentItemsetMining").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/12 14:02:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
conn = sqlite3.connect('tiktok_breadth_first.db')
cursor = conn.cursor()

In [10]:
query = """ 
SELECT hashtag_names 
FROM videos
JOIN follow_relations
ON follow_relations.from_username = videos.reposter_username
WHERE follow_relations.to_username = 'kamalahq'
"""

k_data = cursor.execute(query).fetchall()

In [11]:
k_df = spark.createDataFrame(k_data, ['raw_hash'])

In [9]:
def preprocess(x):
    """
    preprocessing strings with hashtags. commas are delimiters.
    need to filter out non-alphanumeric symbols

    IMPORTANT: filters out 'fyp' from list of hashtags
    """
    hash_list = []
    rough_list = re.split(',', x)
    for hash in rough_list:
        match_alphanum = re.findall(r'[a-zA-Z0-9]+', hash)
        for x in match_alphanum:
            if x.lower() != 'fyp':
                hash_list.append(x.lower())

    return hash_list

In [13]:
k_df = k_df.mapInPandas(preprocess, k_df.schema)

In [20]:
k_df.columns

['raw_hash']

In [21]:
# support: minimum number s of posts an item needs to show up in
s = 100
L1 = (
    k_df.rdd.flatMap(lambda row: [((item,), 1) for item in row.raw_hash])
    .reduceByKey(lambda x, y: x + y)
    .filter(lambda x: x[1] >= s)
    .collect()
)

Lk_1 = (dict(L1))
print("Frequent Singletons (L1)")
print(Lk_1)

26/01/12 14:18:15 WARN TaskSetManager: Stage 5 contains a task of very large size (2301 KiB). The maximum recommended task size is 1000 KiB.
26/01/12 14:18:15 ERROR Executor: Exception in task 7.0 in stage 5.0 (TID 12)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/anitasun/.local/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 1100, in main
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 9) than that in driver 3.11, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:784)
	at org.apache.spark.api.python.PythonRunner$$anon$3.

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/home/anitasun/.local/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 1100, in main
    raise PySparkRuntimeError(
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 9) than that in driver 3.11, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.


In [22]:
conn.close()